# Qaf-ft pilot (Juz 'Amma)
Settings: Accelerator = GPU, Internet = On. Then Run All.

In [ ]:
!pip -q install 'nemo_toolkit[asr]' pyarrow soundfile 2>&1 | tail -2
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import os
os.makedirs('/kaggle/working/qaf/qafft', exist_ok=True); os.makedirs('/kaggle/working/qaf/train', exist_ok=True)
open('/kaggle/working/qaf/qafft/__init__.py','w').close()

In [ ]:
%%writefile /kaggle/working/qaf/qafft/phonetic.py
"""Quran text -> sound-level tokens.

The phonetic script writes a lengthened sound as a repeated symbol (a 4-count
madd is four alif symbols). Repeats are folded into one token "symbol+count",
so a token sequence never has two equal neighbours and stays short enough for
a 12.5 frames/second encoder.
"""
import json
from functools import lru_cache
from itertools import groupby


MAX_RUN = 8


def default_moshaf(madd=4):
    from quran_transcript import MoshafAttributes
    return MoshafAttributes(rewaya="hafs", madd_monfasel_len=min(madd, 5), madd_mottasel_len=max(madd, 4),
                            madd_mottasel_waqf=max(madd, 4), madd_aared_len=madd)


def fold(phonemes: str):
    """'ررَحِۦۦۦۦم' -> ['ر2', 'َ', 'ح', 'ِ', 'ۦ4', 'م']"""
    out = []
    for ch, grp in groupby(phonemes.replace(" ", "")):
        n = min(len(list(grp)), MAX_RUN)
        out.append(ch if n == 1 else f"{ch}{n}")
    return out


def unfold(tokens):
    return "".join(t[0] * (int(t[1:]) if len(t) > 1 else 1) for t in tokens)


@lru_cache(maxsize=None)
def ayah_uthmani(sura: int, aya: int) -> str:
    from quran_transcript import Aya
    return Aya(sura, aya).get().uthmani


def ayah_tokens(sura: int, aya: int, moshaf=None):
    from quran_transcript import quran_phonetizer
    ph = quran_phonetizer(ayah_uthmani(sura, aya), moshaf or default_moshaf(), remove_spaces=True).phonemes
    return fold(ph)


def all_ayahs():
    from quran_transcript import Aya
    for s in range(1, 115):
        for a in range(1, Aya(s, 1).get().num_ayat_in_sura + 1):
            yield s, a


def build_vocab(path):
    """Token list over the whole Quran for madd choices 2, 4 and 6."""
    seen = {}
    for madd in (2, 4, 6):
        m = default_moshaf(madd)
        for s, a in all_ayahs():
            for t in ayah_tokens(s, a, m):
                seen[t] = seen.get(t, 0) + 1
    vocab = sorted(seen, key=lambda t: (t[0], len(t), t))
    json.dump({"tokens": vocab, "counts": seen}, open(path, "w"), ensure_ascii=False, indent=0)
    return vocab


In [ ]:
%%writefile /kaggle/working/qaf/qafft/vocab.json
{
"tokens": [
"ء",
"ا2",
"ا4",
"ا5",
"ا6",
"ب",
"ب2",
"ت",
"ت2",
"ث",
"ث2",
"ج",
"ج2",
"ح",
"ح2",
"خ",
"خ2",
"د",
"د2",
"ذ",
"ذ2",
"ر",
"ر2",
"ز",
"ز2",
"س",
"س2",
"ش",
"ش2",
"ص",
"ص2",
"ض",
"ض2",
"ط",
"ط2",
"ظ",
"ظ2",
"ع",
"ع2",
"غ",
"ـ2",
"ف",
"ف2",
"ق",
"ق2",
"ك",
"ك2",
"ل",
"ل2",
"م",
"م4",
"ن",
"ن3",
"ن4",
"ه",
"ه2",
"و",
"و2",
"و3",
"و5",
"ي",
"ي2",
"ي3",
"ي5",
"َ",
"ُ",
"ِ",
"ٲ",
"ڇ",
"ں3",
"ۜ",
"ۥ2",
"ۥ4",
"ۥ5",
"ۥ6",
"ۦ2",
"ۦ4",
"ۦ5",
"ۦ6",
"۪",
"۾3"
],
"counts": {
"ب": 30594,
"ِ": 142935,
"س": 16005,
"م": 66687,
"ل2": 20754,
"َ": 372186,
"ا2": 82125,
"ه": 44784,
"ر2": 3192,
"ح": 12390,
"ن": 57702,
"ۦ2": 24259,
"ء": 58242,
"ل": 77412,
"د": 15957,
"ُ": 116649,
"ر": 34008,
"ب2": 3855,
"ع": 28176,
"ك": 30855,
"ي": 30181,
"و": 39049,
"د2": 1887,
"ي2": 3240,
"ت": 35724,
"ص2": 1077,
"ط": 3408,
"ق": 20106,
"ص": 5139,
"ذ": 13770,
"غ": 3663,
"ض": 4818,
"ۥ2": 25995,
"ض2": 240,
"ا6": 2573,
"ف": 25827,
"م4": 9600,
"ۦ6": 2351,
"ت2": 2451,
"ز": 4263,
"ڇ": 11511,
"ں3": 15915,
"خ": 7353,
"ا4": 6022,
"ن4": 12264,
"ش": 5580,
"و3": 4381,
"ظ": 2094,
"ي3": 2917,
"۾3": 3174,
"س2": 2031,
"ج": 9597,
"ث": 4125,
"ث2": 114,
"ز2": 534,
"ش2": 792,
"ه2": 126,
"ق2": 993,
"و2": 837,
"ظ2": 465,
"ذ2": 1017,
"ج2": 354,
"ۥ4": 2735,
"ط2": 411,
"ۦ4": 2902,
"ف2": 459,
"ك2": 633,
"خ2": 138,
"ح2": 30,
"ۥ6": 2003,
"ع2": 33,
"۪": 3,
"ـ2": 3,
"ي5": 13,
"ٲ": 3,
"ن3": 9,
"ۜ": 6,
"ا5": 1944,
"ۥ5": 686,
"ۦ5": 587,
"و5": 1
}
}

In [ ]:
%%writefile /kaggle/working/qaf/train/pilot.py
"""Juz 'Amma pilot: fine-tune the Arabic FastConformer encoder to emit sound tokens.

Runs on one GPU (Kaggle T4/P100) or, for a smoke test, on CPU.

  python train/pilot.py --data DIR --out DIR [--steps N] [--smoke]

--data holds parquet shards of the expert-recitation set (one folder per moshaf).
Only segments from suras 78-114 are used. Whole reciters are held out for
validation and test. Training resumes from --out/last.pt if it exists.
"""
import argparse, glob, io, json, math, os, random, sys, time

import numpy as np
import pyarrow.parquet as pq
import soundfile as sf
import torch
import torch.nn as nn

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, ROOT)
from qafft.phonetic import fold

BASE = "nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0"
TEST_MOSHAF = {"5.0", "13.0"}      # never trained on
VAL_MOSHAF = {"19.0"}
FIRST_SURA = 78


class Segments(torch.utils.data.Dataset):
    """Reads audio bytes and phoneme strings straight from parquet row groups."""

    def __init__(self, files, vocab, max_sec=20.0):
        self.vocab, self.items = vocab, []
        for f in files:
            pf = pq.ParquetFile(f)
            for rg in range(pf.num_row_groups):
                t = pf.read_row_group(rg, columns=["start_span", "duration_seconds", "phonemes"]).to_pylist()
                for i, r in enumerate(t):
                    if not r["start_span"] or r["start_span"]["sura_idx"] < FIRST_SURA:
                        continue
                    if r["duration_seconds"] > max_sec or not r["phonemes"]:
                        continue
                    toks = fold(r["phonemes"])
                    if any(x not in vocab for x in toks):
                        continue
                    # 12.5 frames/s doubled by the head = 25 frames/s
                    if len(toks) > 25 * r["duration_seconds"] - 2:
                        continue
                    self.items.append((f, rg, i, r["duration_seconds"], [vocab[x] for x in toks]))
        self._cache = (None, None)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, k):
        f, rg, i, dur, ids = self.items[k]
        if self._cache[0] != (f, rg):
            self._cache = ((f, rg), pq.ParquetFile(f).read_row_group(rg, columns=["audio"]).column("audio").to_pylist())
        wav, sr = sf.read(io.BytesIO(self._cache[1][i]["bytes"]), dtype="float32")
        if wav.ndim > 1:
            wav = wav.mean(1)
        assert sr == 16000, sr
        return torch.from_numpy(wav), torch.tensor(ids)


def collate(batch):
    wavs, ids = zip(*batch)
    wl = torch.tensor([len(w) for w in wavs]); il = torch.tensor([len(i) for i in ids])
    return (nn.utils.rnn.pad_sequence(wavs, batch_first=True), wl,
            nn.utils.rnn.pad_sequence(ids, batch_first=True), il)


def augment(wav, wl):
    """Light, label-safe augmentation: gain and additive noise. No time-warping,
    because stretching changes the madd counts written in the labels."""
    gain = 10 ** (torch.empty(wav.size(0), 1).uniform_(-6, 6) / 20)
    wav = wav * gain
    snr = torch.empty(wav.size(0), 1).uniform_(8, 35)
    rms = wav.pow(2).mean(1, keepdim=True).sqrt().clamp_min(1e-4)
    mask = (torch.rand(wav.size(0), 1) < 0.5).float()
    return wav + mask * torch.randn_like(wav) * rms / (10 ** (snr / 20))


class QafModel(nn.Module):
    def __init__(self, n_tokens, base_path=None):
        super().__init__()
        import nemo.collections.asr as nemo_asr
        m = (nemo_asr.models.ASRModel.restore_from(base_path, map_location="cpu") if base_path
             else nemo_asr.models.ASRModel.from_pretrained(BASE, map_location="cpu"))
        self.pre, self.enc, self.spec = m.preprocessor, m.encoder, m.spec_augmentation
        d = self.enc.d_model if hasattr(self.enc, "d_model") else 512
        self.up = nn.ConvTranspose1d(d, d, kernel_size=2, stride=2)   # 12.5 -> 25 frames/s
        self.out = nn.Linear(d, n_tokens + 1)                          # index n_tokens = blank
        del m

    def forward(self, wav, wl, train=False):
        feats, fl = self.pre(input_signal=wav, length=wl)
        if train and self.spec is not None:
            feats = self.spec(input_spec=feats, length=fl)
        h, hl = self.enc(audio_signal=feats, length=fl)             # (B, D, T)
        h = torch.relu(self.up(h)).transpose(1, 2)
        return self.out(h).log_softmax(-1), hl * 2


def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]


@torch.no_grad()
def evaluate(model, loader, blank, dev, max_batches=0):
    model.eval()
    err = tot = 0
    for b, (wav, wl, ids, il) in enumerate(loader):
        lp, hl = model(wav.to(dev), wl.to(dev))
        best = lp.argmax(-1).cpu()
        for k in range(best.size(0)):
            seq = torch.unique_consecutive(best[k, : hl[k]]).tolist()
            hyp = [x for x in seq if x != blank]
            ref = ids[k, : il[k]].tolist()
            err += edit_distance(hyp, ref); tot += len(ref)
        if max_batches and b + 1 >= max_batches:
            break
    model.train()
    return err / max(1, tot)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--base", default=None, help="local .nemo file; default downloads the base model")
    ap.add_argument("--steps", type=int, default=6000)
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--accum", type=int, default=4)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--head-lr", type=float, default=1e-3)
    ap.add_argument("--eval-every", type=int, default=500)
    ap.add_argument("--smoke", action="store_true")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    random.seed(0); torch.manual_seed(0)
    dev = "cuda" if torch.cuda.is_available() else "cpu"

    tokens = json.load(open(os.path.join(ROOT, "qafft", "vocab.json")))["tokens"]
    vocab = {t: i for i, t in enumerate(tokens)}
    blank = len(tokens)

    def files(pred):
        out = []
        for d in sorted(glob.glob(os.path.join(a.data, "moshaf_*"))):
            mid = os.path.basename(d).split("_", 1)[1]
            if pred(mid):
                out += sorted(glob.glob(os.path.join(d, "*.parquet")))
        return out

    tr = Segments(files(lambda m: m not in TEST_MOSHAF | VAL_MOSHAF), vocab)
    va = Segments(files(lambda m: m in VAL_MOSHAF), vocab)
    te = Segments(files(lambda m: m in TEST_MOSHAF), vocab)
    if a.smoke and not len(va):
        va = tr
    print(f"segments: train {len(tr)} val {len(va)} test {len(te)}", flush=True)
    kw = dict(collate_fn=collate, num_workers=0 if a.smoke else 2)
    # Shuffle row groups, not single rows, so each parquet row group is read once.
    order = sorted(range(len(tr)), key=lambda k: (hash(tr.items[k][:2]) , k))
    tl = torch.utils.data.DataLoader(torch.utils.data.Subset(tr, order), batch_size=a.batch, shuffle=False, **kw)
    vl = torch.utils.data.DataLoader(va, batch_size=a.batch, **kw)

    model = QafModel(len(tokens), a.base).to(dev)
    head = list(model.up.parameters()) + list(model.out.parameters())
    opt = torch.optim.AdamW([{"params": model.enc.parameters(), "lr": a.lr},
                             {"params": head, "lr": a.head_lr}], weight_decay=1e-3)
    warm = 300
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm))
    scaler = torch.amp.GradScaler(dev, enabled=dev == "cuda")
    ctc = nn.CTCLoss(blank=blank, zero_infinity=True)
    step, best = 0, 1e9
    last = os.path.join(a.out, "last.pt")
    if os.path.exists(last):
        ck = torch.load(last, map_location=dev)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        step, best = ck["step"], ck["best"]
        print("resumed at step", step, flush=True)

    model.train()
    t0, run = time.time(), 0.0
    while step < a.steps:
        for n, (wav, wl, ids, il) in enumerate(tl):
            wav = augment(wav, wl)
            with torch.autocast(dev, enabled=dev == "cuda"):
                lp, hl = model(wav.to(dev), wl.to(dev), train=True)
            loss = ctc(lp.float().transpose(0, 1), ids.to(dev), hl, il.to(dev)) / a.accum
            scaler.scale(loss).backward()
            run += loss.item() * a.accum
            if (n + 1) % a.accum:
                continue
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            step += 1
            if step % 20 == 0 or a.smoke:
                print(f"step {step} loss {run / (a.accum * (20 if not a.smoke else 1)):.3f} "
                      f"{(time.time() - t0) / step:.1f}s/step", flush=True)
                run = 0.0
            if step % a.eval_every == 0 or step >= a.steps:
                per = evaluate(model, vl, blank, dev, max_batches=2 if a.smoke else 60)
                print(f"step {step} val token error rate {per:.4f}", flush=True)
                ck = dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                          step=step, best=min(best, per), tokens=tokens)
                torch.save(ck, last)
                if per < best:
                    best = per
                    torch.save(dict(model=model.state_dict(), tokens=tokens, step=step, per=per),
                               os.path.join(a.out, "best.pt"))
            if step >= a.steps:
                break
    if len(te):
        per = evaluate(model, torch.utils.data.DataLoader(te, batch_size=a.batch, **kw), blank, dev)
        print(f"held-out reciters token error rate {per:.4f}", flush=True)
        json.dump(dict(test_token_error_rate=per, step=step), open(os.path.join(a.out, "result.json"), "w"))


if __name__ == "__main__":
    main()


In [ ]:
# Last shard of every moshaf holds suras 78-114.
import collections, requests
from huggingface_hub import hf_hub_download
REPO = 'obadx/muaalem-annotated-v3'
files = [s['rfilename'] for s in requests.get(f'https://huggingface.co/api/datasets/{REPO}').json()['siblings']]
by = collections.defaultdict(list)
for f in files:
    if f.startswith('moshaf_') and f.endswith('.parquet'): by[f.split('/')[0]].append(f)
for m, fs in sorted(by.items()):
    p = hf_hub_download(REPO, sorted(fs)[-1], repo_type='dataset', local_dir='/kaggle/temp/data')
    print(m, os.path.getsize(p) // 2**20, 'MB')

In [ ]:
!cd /kaggle/working/qaf && python train/pilot.py --data /kaggle/temp/data --out /kaggle/working/pilot --steps 3000 --batch 8 --accum 2 --eval-every 250 2>&1 | grep -v -E 'NeMo [WI]|^\s*$|Cuda graphs|Reason:'

In [ ]:
!ls -la /kaggle/working/pilot; cat /kaggle/working/pilot/result.json